<a href="https://colab.research.google.com/github/ottrindade1963/analise-industrialXL/blob/main/Pipeline_Africa_MO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline de Análise Industrial - África e Médio Oriente

## Passos 1 a 10 com Geração Automática de Metadados

Este notebook executa o fluxo completo de análise de dados para 37 países da África e Médio Oriente, utilizando dados do Banco Mundial (WDI + WGI).

**Características:**
- Clonagem automática do repositório GitHub
- Extração automática via API (WDI + WGI)
- Agregação INNER JOIN + Dados Sintéticos (500 anos)
- Engenharia de Features avançada (lags, MA, deltas, interações)
- 7 Modelos (5 Clássicos + 2 Bayesianos)
- Interpretabilidade SHAP + Análise Geográfica
- **Sincronização simultânea de TODOS os ficheiros no Google Drive**
- **Visualização inline de TODAS as imagens geradas em cada passo**

## 0. Configuração Inicial do Colab

In [ ]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive montado em /content/drive")

In [ ]:
# Instalar dependências
!pip install -q wbgapi pmdarima xgboost shap geopandas pymc arviz tensorflow scipy
print("✓ Dependências instaladas")

In [ ]:
# Clonar repositório GitHub
!git clone https://github.com/ottrindade1963/analise-industrialXL.git /content/repo
print("✓ Repositório clonado do GitHub")

In [ ]:
import os
import sys
import time
import shutil
import glob
from datetime import datetime
from pathlib import Path
from IPython.display import display, Image, HTML

# Configurar paths
REPO_DIR = '/content/repo'  # Repositório clonado do GitHub
PIPELINE_DIR = os.path.join(REPO_DIR, 'pipeline_africa_mo')  # Subdiretório do pipeline
DRIVE_DIR = '/content/drive/MyDrive/pipeline_africa_mo_resultados'  # Resultados no Drive

# Verificar se o pipeline está no subdiretório ou na raiz
if not os.path.exists(PIPELINE_DIR):
    PIPELINE_DIR = REPO_DIR  # Se estiver na raiz

os.chdir(PIPELINE_DIR)
sys.path.insert(0, PIPELINE_DIR)

# Criar diretório de resultados no Drive
os.makedirs(DRIVE_DIR, exist_ok=True)

print(f"✓ Diretório de trabalho: {PIPELINE_DIR}")
print(f"✓ Diretório de resultados (Drive): {DRIVE_DIR}")
print(f"\n  Ficheiros do pipeline:")
for f in sorted(os.listdir(PIPELINE_DIR)):
    if f.endswith('.py') or f.endswith('.md'):
        print(f"    {f}")

In [ ]:
# Função auxiliar para sincronizar TODOS os ficheiros gerados com o Drive
def sincronizar_todos_drive():
    """
    Sincroniza TODOS os diretórios de resultados com o Google Drive.
    Chamada após cada passo para backup simultâneo.
    """
    diretorio_saida = [
        'dados_brutos',
        'dados_limpos',
        'dados_agregados',
        'dados_sinteticos',
        'dados_engenharia',
        'eda_brutos',
        'eda_agregados',
        'eda_engenharia',
        'modelos_treinados',
        'resultados_avaliacao',
        'analise_estrategias',
        'shap_analysis',
        'analise_geografica',
        'analise_avancada',
        'metadados'
    ]

    ficheiros_sincronizados = 0

    for dir_name in diretorio_saida:
        origem = os.path.join(PIPELINE_DIR, dir_name)
        if not os.path.exists(origem):
            continue

        destino = os.path.join(DRIVE_DIR, dir_name)
        os.makedirs(destino, exist_ok=True)

        # Sincronizar ficheiros (incluindo subdirectórios)
        for root, dirs, files in os.walk(origem):
            rel_path = os.path.relpath(root, origem)
            dest_root = os.path.join(destino, rel_path) if rel_path != '.' else destino
            os.makedirs(dest_root, exist_ok=True)
            for f in files:
                src = os.path.join(root, f)
                dst = os.path.join(dest_root, f)
                shutil.copy2(src, dst)
                ficheiros_sincronizados += 1

    return ficheiros_sincronizados


def mostrar_imagens(diretorios, titulo=None):
    """
    Mostra inline TODAS as imagens PNG encontradas nos directórios especificados.
    Pesquisa recursivamente em subdirectórios.

    Args:
        diretorios: lista de caminhos absolutos ou relativos a PIPELINE_DIR
        titulo: título opcional para o bloco de imagens
    """
    imagens = []
    for d in diretorios:
        # Resolver caminho absoluto
        if not os.path.isabs(d):
            d = os.path.join(PIPELINE_DIR, d)
        if not os.path.exists(d):
            continue
        # Buscar PNGs recursivamente
        for root, dirs, files in os.walk(d):
            for f in sorted(files):
                if f.lower().endswith('.png'):
                    imagens.append(os.path.join(root, f))

    if not imagens:
        print("  (Nenhuma imagem gerada neste passo)")
        return

    if titulo:
        display(HTML(f'<h3>📊 {titulo} ({len(imagens)} imagens)</h3>'))
    else:
        display(HTML(f'<h3>📊 Visualizações geradas ({len(imagens)} imagens)</h3>'))

    for img_path in imagens:
        # Mostrar nome do ficheiro como legenda
        nome_ficheiro = os.path.basename(img_path)
        subdir = os.path.basename(os.path.dirname(img_path))
        display(HTML(f'<hr><b>{subdir}/{nome_ficheiro}</b>'))
        display(Image(filename=img_path, width=900))

    print(f"\n  ✓ {len(imagens)} imagens visualizadas")


print("✓ Funções auxiliares definidas (sincronização + visualização de imagens)")

---
## 1. Extração de Dados (WDI + WGI)

In [ ]:
print("\n" + "="*70)
print("  PASSO 1: EXTRAÇÃO DE DADOS VIA API")
print("="*70)

t0 = time.time()
from passo1_extracao import executar_passo1
executar_passo1()

tempo_p1 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p1:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")
# Passo 1 não gera imagens (apenas CSVs de dados brutos)

---
## 2. EDA dos Dados Brutos

In [ ]:
print("\n" + "="*70)
print("  PASSO 2: EDA DOS DADOS BRUTOS")
print("="*70)

t0 = time.time()
from passo2_eda_brutos import executar_passo2
executar_passo2()

tempo_p2 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p2:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")

In [ ]:
# Visualizar imagens geradas no Passo 2
mostrar_imagens(['eda_brutos'], titulo='Passo 2 - EDA Dados Brutos')

---
## 2.1. Limpeza de Dados

In [ ]:
print("\n" + "="*70)
print("  PASSO 2.1: LIMPEZA DE DADOS")
print("="*70)

t0 = time.time()
from passo2_1_limpeza import executar_passo2_1
executar_passo2_1()

tempo_p2_1 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p2_1:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")

In [ ]:
# Visualizar imagens geradas no Passo 2.1 (EDA WDI + WGI limpos)
mostrar_imagens(
    [os.path.join('dados_limpos', 'eda_wdi'), os.path.join('dados_limpos', 'eda_wgi')],
    titulo='Passo 2.1 - EDA após Limpeza (WDI + WGI)'
)

---
## 2.2. Agregação INNER JOIN + Dados Sintéticos

In [ ]:
print("\n" + "="*70)
print("  PASSO 2.2: AGREGAÇÃO INNER JOIN + DADOS SINTÉTICOS (500 ANOS)")
print("="*70)

t0 = time.time()
from passo2_2_agregacao_sinteticos import executar_passo2_2
executar_passo2_2()

tempo_p2_2 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p2_2:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")
# Passo 2.2 não gera imagens (apenas CSVs de dados agregados e sintéticos)

---
## 2.3. EDA Agregados + Sintéticos

In [ ]:
print("\n" + "="*70)
print("  PASSO 2.3: EDA AGREGADOS + SINTÉTICOS")
print("="*70)

t0 = time.time()
from passo2_3_eda_agregados import executar_passo2_3
executar_passo2_3()

tempo_p2_3 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p2_3:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")

In [ ]:
# Visualizar imagens geradas no Passo 2.3
mostrar_imagens(['eda_agregados'], titulo='Passo 2.3 - EDA Agregados e Sintéticos')

---
## 3. Engenharia de Features

In [ ]:
print("\n" + "="*70)
print("  PASSO 3: ENGENHARIA DE FEATURES AVANÇADA")
print("="*70)

t0 = time.time()
from passo3_engenharia_features import executar_passo3
executar_passo3()

tempo_p3 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p3:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")


In [ ]:
# Visualizar imagens geradas no Passo 3 (PCA e análise de features)
mostrar_imagens(['eda_engenharia'], titulo='Passo 3 - Engenharia de Features (PCA e Análise)')

---
## 4. Treinamento de 7 Modelos

In [ ]:
print("\n" + "="*70)
print("  PASSO 4: TREINAMENTO DE 7 MODELOS")
print("  (5 Clássicos: RF, XGBoost, GradientBoosting, SARIMAX, LSTM)")
print("  (2 Bayesianos: PartialPooling, CompletePooling)")
print("="*70)

t0 = time.time()
from passo4_treino_modelos import executar_passo4
executar_passo4()

tempo_p4 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p4:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")
# Passo 4 não gera imagens (apenas modelos .pkl e metadados)

---
## 5. Avaliação de Performance

In [ ]:
print("\n" + "="*70)
print("  PASSO 5: AVALIAÇÃO DE PERFORMANCE")
print("="*70)

t0 = time.time()
from passo5_avaliacao import executar_passo5
executar_passo5()

tempo_p5 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p5:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")

In [ ]:
# Visualizar imagens geradas no Passo 5
mostrar_imagens(['resultados_avaliacao'], titulo='Passo 5 - Avaliação de Performance')

### **SEGUNDA PARTE (OS FICHEIROS TEMPORÁRIOS PRECISAM SER RESTURADOS**

**Instalando dependência**

In [ ]:
!pip install gdown -q
!pip install ruptures


**FORÇAR NOVA CLONAGEM**

In [ ]:
import os

# 1. Vai para um diretório seguro (o /content sempre existe)
os.chdir('/content')

# 2. Remove qualquer resquício antigo (ignora erros se não existir)
!rm -rf /content/repo

# 3. Clona novamente
!git clone https://github.com/ottrindade1963/analise-industrialXL.git /content/repo

# 4. Entra no diretório clonado
os.chdir('/content/repo')

# 5. Verifica o commit desejado (use o hash real, sem <>)
!git checkout 434a91d

print("✓ Repositório clonado e posicionado no commit 434a91d")

**RESTAURAR FICHEIROS DO DRIVE**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import urllib.request

url = 'https://raw.githubusercontent.com/ottrindade1963/analise-industrialXL/main/colab_auto_sync_github.py'

exec(urllib.request.urlopen(url ).read())


**VERIFICAÇÃO E ORDENAÇÃO DOS CAMINHO PARA O PASSO6**

In [ ]:
import os
import sys

# Corrigir caminhos para o Colab
sys.path.insert(0, '/content/repo/pipeline_africa_mo')
import config_global as config

# Sobrescrever caminhos
config.DADOS_ENGENHARIA_DIR = '/content/repo/pipeline_africa_mo/dados_engenharia'
config.MODELOS_DIR = '/content/repo/pipeline_africa_mo/modelos'
config.ESTRATEGIAS_DIR = '/content/repo/pipeline_africa_mo/analise_estrategias'

# Verificar
print("Caminhos corrigidos:")
print(f"  Dados: {config.DADOS_ENGENHARIA_DIR}")
print(f"  Modelos: {config.MODELOS_DIR}")

# Agora execute o passo 6
from passo6_estrategias import executar_passo6
executar_passo6()


---
## 6. Análise de Estratégias

In [ ]:
import time
import os
import sys

print("\n" + "="*70)
print("  PASSO 6: ANÁLISE DE ESTRATÉGIAS")
print("="*70)

t0 = time.time()
from passo6_estrategias import executar_passo6
executar_passo6()

tempo_p6 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p6:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
#n_sync = sincronizar_todos_drive()
#print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")

CÓDIGO PARA FORÇAR A VISUALIZAÇÃO NESTA SESSÃO

In [ ]:
import os
from IPython.display import display, Image, HTML

def mostrar_imagens(diretorios, titulo=None):
    """
    Mostra inline TODAS as imagens PNG encontradas nos directórios especificados.
    Compatível com Colab e sandbox.

    Args:
        diretorios: Lista de nomes de directórios (ex: ['analise_estrategias', 'shap_analysis'])
        titulo: Título opcional para a galeria
    """
    PIPELINE_DIR = '/content/repo/pipeline_africa_mo'  # Ajuste se necessário

    imagens = []

    for d in diretorios:
        # Resolver caminho absoluto
        if not os.path.isabs(d):
            d = os.path.join(PIPELINE_DIR, d)

        if not os.path.exists(d):
            print(f"  ⚠️ Directório não encontrado: {d}")
            continue

        # Buscar PNGs recursivamente
        for root, dirs, files in os.walk(d):
            for f in sorted(files):
                if f.lower().endswith('.png'):
                    imagens.append(os.path.join(root, f))

    if not imagens:
        print("  (Nenhuma imagem gerada neste passo)")
        return

    # Mostrar título
    if titulo:
        display(HTML(f'<h2>📊 {titulo}</h2>'))
        display(HTML(f'<p><b>Total: {len(imagens)} imagens</b></p>'))
    else:
        display(HTML(f'<h2>📊 Visualizações geradas ({len(imagens)} imagens)</h2>'))

    # Mostrar cada imagem
    for i, img_path in enumerate(imagens, 1):
        nome_ficheiro = os.path.basename(img_path)
        subdir = os.path.basename(os.path.dirname(img_path))

        display(HTML(f'<hr><p><b>[{i}/{len(imagens)}] {subdir}/{nome_ficheiro}</b></p>'))
        display(Image(filename=img_path, width=900))

    print(f"\n✓ {len(imagens)} imagens visualizadas com sucesso!")

print("✓ Função mostrar_imagens() definida com sucesso!")


In [ ]:
# Visualizar Passo 6 com informações
print("\n" + "="*70)
print("  VISUALIZANDO PASSO 6 - ANÁLISE DE ESTRATÉGIAS")
print("="*70)

mostrar_imagens(['analise_estrategias'], titulo='Passo 6 - Análise de Estratégias')

print("\n✓ Visualização concluída!")


---
## 7. Interpretabilidade (SHAP)

In [ ]:
import os
import sys
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BASE_DIR = '/content/pipeline_africa_mo'   # ajuste se estiver noutro local
if not os.path.exists(BASE_DIR):
    print(f"ERRO: {BASE_DIR} não existe!")
    # Se estiver no Drive, use algo como '/content/drive/MyDrive/pipeline_africa_mo'

In [ ]:
sys.path.insert(0, BASE_DIR)
os.chdir(BASE_DIR)   # muda para o diretório do projeto

print("Conteúdo do diretório base:")
!ls -la

print("\nConteúdo da pasta modelos_treinados:")
!ls -la modelos_treinados/

In [ ]:
print("\nConteúdo da pasta dados_engenharia (se existir):")
if os.path.exists('dados_engenharia'):
    !ls -la dados_engenharia/
else:
    print("Pasta 'dados_engenharia' não encontrada no diretório base.")
    print("O Passo 7 precisa desses CSVs para carregar os dados.")

In [ ]:
!pip install shap -q

In [ ]:
import config_global

# Forçar o caminho dos modelos e dos dados de engenharia
config_global.MODELOS_DIR = os.path.join(BASE_DIR, 'modelos_treinados')
config_global.DADOS_ENGENHARIA_DIR = os.path.join(BASE_DIR, 'dados_engenharia')

print("MODELOS_DIR =", config_global.MODELOS_DIR)
print("DADOS_ENGENHARIA_DIR =", config_global.DADOS_ENGENHARIA_DIR)

# Importar e executar
from passo7_shap import executar_passo7

executar_passo7()

In [ ]:
import os
import sys
import glob
import pickle
import time
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. PROCURAR OS FICHEIROS NECESSÁRIOS
# ============================================================
print("🔍 Procurando ficheiros '_features.csv' e modelos '.pkl'...")

csv_files = []
pkl_files = []

# Pesquisar em /content e /content/drive (se montado)
search_paths = ['/content', '/content/drive']
for path in search_paths:
    if os.path.exists(path):
        for root, dirs, files in os.walk(path):
            for f in files:
                if f.endswith('_features.csv'):
                    csv_files.append(os.path.join(root, f))
                elif f.endswith('.pkl') and ('modelo_' in f):
                    pkl_files.append(os.path.join(root, f))

if not csv_files:
    print("❌ Nenhum ficheiro '_features.csv' encontrado. Monte o Google Drive e certifique-se de que os dados estão lá.")
    sys.exit(1)

if not pkl_files:
    print("❌ Nenhum modelo '.pkl' encontrado. Monte o Google Drive e certifique-se de que os modelos treinados estão presentes.")
    sys.exit(1)

print(f"✅ Encontrados {len(csv_files)} ficheiros CSV e {len(pkl_files)} modelos .pkl")
print("\n📁 CSVs encontrados:")
for f in csv_files[:5]:
    print(f"   {f}")
print("\n📁 Modelos encontrados (amostra):")
for f in pkl_files[:5]:
    print(f"   {f}")

# ============================================================
# 2. DETERMINAR OS DIRETÓRIOS BASE
# ============================================================
# Vamos extrair o diretório comum onde estão os CSVs (ex: .../dados_engenharia)
csv_dir = os.path.dirname(csv_files[0])
# Para os modelos, o diretório deve ser o que contém os .pkl (ex: .../modelos_treinados)
pkl_dir = os.path.dirname(pkl_files[0])

# Mas é possível que os CSVs estejam num subdiretório 'dados_engenharia' e os modelos noutro.
# Vamos tentar encontrar o diretório base (pipeline_africa_mo) a partir dos caminhos.
base_candidates = []
for f in csv_files + pkl_files:
    parts = os.path.normpath(f).split(os.sep)
    # Procurar por 'pipeline_africa_mo' no caminho
    if 'pipeline_africa_mo' in parts:
        idx = parts.index('pipeline_africa_mo')
        base_candidates.append(os.sep.join(parts[:idx+1]))

if base_candidates:
    BASE_DIR = base_candidates[0]
else:
    # Fallback: usar o diretório do primeiro CSV
    BASE_DIR = os.path.dirname(csv_dir)

print(f"\n📍 Diretório base identificado: {BASE_DIR}")

# Ajustar os diretórios específicos
DADOS_ENG_DIR = csv_dir
MODELOS_DIR = pkl_dir

print(f"📂 DADOS_ENGENHARIA_DIR = {DADOS_ENG_DIR}")
print(f"📂 MODELOS_DIR = {MODELOS_DIR}")

# ============================================================
# 3. CONFIGURAR O AMBIENTE E IMPORTAR O PASSO 7
# ============================================================
# Adicionar o BASE_DIR ao sys.path para importar config_global e passo7_shap
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)
os.chdir(BASE_DIR)

import config_global

# Sobrescrever os diretórios com os caminhos encontrados
config_global.DADOS_ENGENHARIA_DIR = DADOS_ENG_DIR
config_global.MODELOS_DIR = MODELOS_DIR
config_global.SHAP_DIR = os.path.join(BASE_DIR, 'shap_analysis')

print(f"\n✅ Configuração final:")
print(f"   DADOS_ENGENHARIA_DIR = {config_global.DADOS_ENGENHARIA_DIR}")
print(f"   MODELOS_DIR = {config_global.MODELOS_DIR}")
print(f"   SHAP_DIR = {config_global.SHAP_DIR}")

# Verificação rápida: deve listar os CSVs agora
print("\n🔎 Verificando se o diretório de dados contém os CSVs:")
if os.path.exists(config_global.DADOS_ENGENHARIA_DIR):
    print("   Ficheiros encontrados:", os.listdir(config_global.DADOS_ENGENHARIA_DIR)[:5])
else:
    print("   ERRO: diretório não existe após configuração.")

# ============================================================
# 4. EXECUTAR O PASSO 7 (SHAP)
# ============================================================
from passo7_shap import executar_passo7

print("\n" + "="*70)
print("  INICIANDO PASSO 7 - SHAP E FEATURE IMPORTANCE")
print("="*70)

t0 = time.time()
executar_passo7()
tempo = time.time() - t0
print(f"\n⏱ Tempo total: {tempo:.1f} segundos")

In [ ]:
import os
import config_global
print("SHAP_DIR =", config_global.SHAP_DIR)
print("Arquivos gerados:")
for f in sorted(os.listdir(config_global.SHAP_DIR)):
    print(f"  - {f}")

In [ ]:
from IPython.display import Image, display, HTML
import os

shap_dir = config_global.SHAP_DIR
png_files = [f for f in os.listdir(shap_dir) if f.endswith('.png')]

print(f"📊 {len(png_files)} imagens PNG encontradas:\n")
for png in sorted(png_files):
    display(HTML(f"<b>{png}</b>"))
    display(Image(filename=os.path.join(shap_dir, png)))
    print("\n" + "-"*80 + "\n")

---
## 8. Análise Geográfica

In [ ]:
print("\n" + "="*70)
print("  PASSO 8: ANÁLISE GEOGRÁFICA")
print("="*70)

t0 = time.time()
from passo8_geografica import executar_passo8
executar_passo8()

tempo_p8 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p8:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")

In [ ]:
# Visualizar imagens geradas no Passo 8
mostrar_imagens(['analise_geografica'], titulo='Passo 8 - Análise Geográfica')

---
## 9. Análises Avançadas

In [ ]:
import os
import shutil
import subprocess

# ============================================================
# CONFIGURAÇÕES (edite conforme o seu caso)
# ============================================================
REPO_URL = "https://github.com/ottrindade1963/analise-industrialXL.git"   # <-- MUDE AQUI
REPO_NAME = "pipeline_africa_mo"          # nome da pasta local
BASE_PATH = f"/content/{REPO_NAME}"

# Se quiser usar o Google Drive (persistente), descomente:
# from google.colab import drive
# drive.mount('/content/drive')
# BASE_PATH = f"/content/drive/MyDrive/{REPO_NAME}"

# ============================================================
# 1. REMOVER O REPOSITÓRIO ANTIGO (OPÇÃO RADICAL)
# ============================================================
# Escolha a opção que desejar:
#   Opção A: reset hard + pull (mantém a pasta, mas força sincronia)
#   Opção B: remove tudo e clona novamente (mais seguro para evitar conflitos)

print("Atualizando repositório...")

if os.path.exists(BASE_PATH):
    print(f"Repositório já existe em {BASE_PATH}. Forçando atualização...")

    # Opção B: remover e clonar do zero (recomendado para garantir ficheiros novos)
    print("   Removendo diretório antigo...")
    shutil.rmtree(BASE_PATH)   # cuidado: remove todos os ficheiros locais não commitados
    print("   Diretório removido.")
else:
    print(f"Repositório não encontrado em {BASE_PATH}. Será clonado.")

# ============================================================
# 2. CLONAR O REPOSITÓRIO FRESCO
# ============================================================
print(f"Clonando {REPO_URL} para {BASE_PATH}...")
result = subprocess.run(["git", "clone", REPO_URL, BASE_PATH], capture_output=True, text=True)
if result.returncode != 0:
    print("ERRO ao clonar:")
    print(result.stderr)
    raise Exception("Falha na clonagem. Verifique o URL e a rede.")
else:
    print("Clone bem-sucedido.")

# ============================================================
# 3. (Opcional) Se você fez alterações locais que quer preservar,
#    faça backup antes de remover. Mas como é no Colab, geralmente
#    não há alterações locais importantes.
# ============================================================

# ============================================================
# 4. Verificar se os ficheiros esperados existem
# ============================================================
print("\nVerificando ficheiros importantes:")
essential_files = ["config_global.py", "passo7_shap.py", "passo9_avancada.py"]
for f in essential_files:
    path = os.path.join(BASE_PATH, f)
    if os.path.exists(path):
        print(f"  ✅ {f} encontrado")
    else:
        print(f"  ❌ {f} NÃO encontrado!")

# ============================================================
# 5. Adicionar o caminho ao sys.path e importar configuração
# ============================================================
import sys
if BASE_PATH not in sys.path:
    sys.path.insert(0, BASE_PATH)
os.chdir(BASE_PATH)

import config_global
print(f"\nDiretório de modelos: {config_global.MODELOS_DIR}")
print(f"Diretório de dados: {config_global.DADOS_ENGENHARIA_DIR}")

print("\n✅ Atualização concluída. Agora pode executar os passos normalmente.")

In [ ]:
def sincronizar_todos_drive():
    """Função placeholder para compatibilidade."""
    print("Sincronização já feita pelo passo 9.")
    return 0

In [ ]:
print("\n" + "="*70)
print("  PASSO 9: ANÁLISES AVANÇADAS")
print("="*70)

t0 = time.time()
from passo9_avancada import executar_passo9
executar_passo9()

tempo_p9 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p9:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")

In [ ]:
from IPython.display import Image, display
import os
import config_global

png_dir = config_global.AVANCADA_DIR
png_files = [f for f in os.listdir(png_dir) if f.endswith('.png')]

print(f"📊 {len(png_files)} gráficos encontrados:\n")
for png in sorted(png_files):
    print(f"▶ {png}")
    display(Image(filename=os.path.join(png_dir, png)))
    print("-" * 80)

**10. INOVAÇÕES**

In [ ]:
import sys
import os

# Caminho onde está o seu pipeline (ajuste se necessário)
projeto_dir = '/content/pipeline_africa_mo'

if projeto_dir not in sys.path:
    sys.path.insert(0, projeto_dir)

# Verificar se o ficheiro existe
if os.path.exists(os.path.join(projeto_dir, 'passo10_inovacoes_mestrado.py')):
    print("✅ Módulo encontrado")
else:
    print("❌ Ficheiro não encontrado. Verifique o caminho.")

**LOCALIZAR AUTOMATICAMENTE OS DIRETÓRIOS DO PASSO 10**

In [ ]:
import os
import sys
import pandas as pd
import pickle
import glob

# ============================================================
# LOCALIZAR AUTOMATICAMENTE OS DIRETÓRIOS
# ============================================================
print("🔍 Localizando dados e modelos...")

# 1. Procurar ficheiros 'agregado_features.csv'
caminhos_csv = glob.glob('/content/**/agregado_features.csv', recursive=True)
if not caminhos_csv:
    caminhos_csv = glob.glob('/content/drive/**/agregado_features.csv', recursive=True)

if not caminhos_csv:
    print("❌ Nenhum 'agregado_features.csv' encontrado.")
    print("Monte o Google Drive e certifique-se de que os dados estão presentes.")
    # Montar drive se necessário
    from google.colab import drive
    drive.mount('/content/drive')
    caminhos_csv = glob.glob('/content/drive/**/agregado_features.csv', recursive=True)

if caminhos_csv:
    csv_path = caminhos_csv[0]
    dados_dir = os.path.dirname(csv_path)
    base_dir = os.path.dirname(dados_dir)  # assume que dados_engenharia está dentro do projeto
    print(f"✅ Dados encontrados em: {dados_dir}")
    print(f"   Base do projeto: {base_dir}")
else:
    raise FileNotFoundError("Não foi possível localizar os dados. Verifique se os CSVs existem.")

# 2. Procurar modelos .pkl (pelo menos um)
modelos_dir_candidates = glob.glob(f'{base_dir}/**/modelos_treinados', recursive=True)
if not modelos_dir_candidates:
    modelos_dir_candidates = glob.glob('/content/**/modelos_treinados', recursive=True)

if modelos_dir_candidates:
    modelos_dir = modelos_dir_candidates[0]
    print(f"✅ Modelos encontrados em: {modelos_dir}")
else:
    print("⚠️ Nenhum modelo encontrado. A Sugestão 2 e 3 podem não funcionar.")
    modelos_dir = None

# ============================================================
# FORÇAR CONFIG_GLOBAL COM OS CAMINHOS ENCONTRADOS
# ============================================================
import config_global as config
config.DADOS_ENGENHARIA_DIR = dados_dir
config.MODELOS_DIR = modelos_dir if modelos_dir else config.MODELOS_DIR
config.BASE_DIR = base_dir

print(f"\nConfiguração atualizada:")
print(f"  DADOS_ENGENHARIA_DIR = {config.DADOS_ENGENHARIA_DIR}")
print(f"  MODELOS_DIR = {config.MODELOS_DIR}")
print(f"  BASE_DIR = {config.BASE_DIR}")

# ============================================================
# EXECUTAR PASSO 10
# ============================================================
# Garantir que o diretório do projeto está no sys.path
if base_dir not in sys.path:
    sys.path.insert(0, base_dir)
os.chdir(base_dir)

# Importar e executar
from passo10_inovacoes_mestrado import executar_passo10
executar_passo10()

**Executar Passo 10**

In [ ]:
from passo10_inovacoes_mestrado import executar_passo10
executar_passo10()

**Visualizar passo 10**

In [ ]:
import os
from IPython.display import Image, display

# Pasta onde o Passo 10 guarda os ficheiros
PASTA = '/content/pipeline_africa_mo/inovacoes_mestrado'

print("📁 Conteúdo da pasta inovacoes_mestrado:")
for f in sorted(os.listdir(PASTA)):
    print(f"  - {f}")

print("\n" + "="*60)
print("🖼️ GRÁFICOS GERADOS")
print("="*60)

# Mostrar cada imagem PNG
for png in sorted([f for f in os.listdir(PASTA) if f.endswith('.png')]):
    print(f"\n▶ {png}")
    display(Image(filename=os.path.join(PASTA, png)))

**visualizar os gráficos**

In [ ]:
!zip -r inovacoes_mestrado.zip /content/pipeline_africa_mo/inovacoes_mestrado

In [ ]:
from google.colab import files
files.download('inovacoes_mestrado.zip')